# Lesson 03c — Advanced RAG Patterns
**Domain:** Scientific Literature  
**Dataset:** arXiv abstracts (HF) + Wikipedia (20220301.en, 500 articles)  
**Time estimate:** ~2.5 h

## Learning objectives
- **HyDE**: generate a hypothetical document, embed it, retrieve by similarity
- **Multi-Query**: 3 reframings → 3 retrievals → merge with Reciprocal Rank Fusion (RRF)
- **Contextual retrieval** (Anthropic 2024): prepend chunk context before embedding
- Measure Recall@3 improvement for each technique

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
from openai import OpenAI
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.utils import embedding_functions
from llm_checker import check, hint, evaluate, progress

local_client = OpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio")
MODEL        = "local-model"
embed_model  = SentenceTransformer("all-MiniLM-L6-v2")

def llm(prompt, system=None, max_tokens=200, temperature=0.3):
    messages = ([{"role":"system","content":system}] if system else []) + [{"role":"user","content":prompt}]
    r = local_client.chat.completions.create(model=MODEL, messages=messages,
                                              max_tokens=max_tokens, temperature=temperature)
    return r.choices[0].message.content.strip()

def cosine_similarity(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9))

print("✅ Setup complete")

## Rebuild ChromaDB collection (arXiv 200 abstracts)

In [ ]:
from datasets import load_dataset

arxiv_ds = load_dataset("gfissore/arxiv-abstracts-2021", split="train").select(range(2000))
abstracts = [
    {"id": str(i), "title": row.get("title",""), "text": row.get("abstract","") or row.get("text","")}
    for i, row in enumerate(arxiv_ds)
]

ef = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")
chroma_client = chromadb.Client()
try:
    chroma_client.delete_collection("arxiv_03c")
except:
    pass
collection = chroma_client.create_collection("arxiv_03c", embedding_function=ef)

for i in range(0, 200, 50):
    batch = abstracts[i:i+50]
    collection.add(
        documents=[a["text"] for a in batch],
        metadatas=[{"title": a["title"]} for a in batch],
        ids=[a["id"] for a in batch],
    )
print(f"✅ Indexed {collection.count()} abstracts")

# Ground truth pairs (reuse from 03a-3 pattern)
ground_truth_20 = [
    {"question": llm(f"One short question answered by this abstract:\n{abstracts[i]['text'][:400]}", max_tokens=40),
     "expected_id": str(i)}
    for i in range(20)
]
print(f"✅ {len(ground_truth_20)} ground-truth Q&A pairs built")

def recall_at_k(pairs, k, query_fn):
    hits = 0
    for pair in pairs:
        top_ids = query_fn(pair["question"], k)
        if pair["expected_id"] in top_ids:
            hits += 1
    return hits / len(pairs)

def vanilla_query(question, k=3):
    res = collection.query(query_texts=[question], n_results=k)
    return res["ids"][0]

r3_vanilla = recall_at_k(ground_truth_20, 3, vanilla_query)
print(f"Vanilla Recall@3 baseline: {r3_vanilla:.0%}")

---
## 🔵 EXAMPLE — Ex 03c-1: HyDE implementation

For a question, generate a hypothetical answer passage, embed it, and retrieve from arXiv.
Compare top-3 results vs embedding the question directly.

In [ ]:
def hyde_query(question: str, k: int = 3) -> list:
    """HyDE: generate a hypothetical document, embed it, retrieve."""
    hypothetical = llm(
        f"Write a short scientific paper excerpt (3-4 sentences) that answers: {question}",
        max_tokens=150, temperature=0.5
    )
    res = collection.query(query_texts=[hypothetical], n_results=k)
    return res["ids"][0]

test_question = "What are the main applications of transformer models in genomics?"

vanilla_top3 = vanilla_query(test_question)
hyde_top3    = hyde_query(test_question)

print(f"Question: {test_question}")
print(f"\nVanilla top-3: {vanilla_top3}")
print(f"HyDE    top-3: {hyde_top3}")
print(f"\nDifferences: {set(hyde_top3) - set(vanilla_top3)} (new IDs from HyDE)")
print("Comment: HyDE finds different chunks — sometimes more relevant, sometimes noisier.")

r3_hyde = recall_at_k(ground_truth_20, 3, hyde_query)
print(f"\nVanilla Recall@3 : {r3_vanilla:.0%}")
print(f"HyDE    Recall@3 : {r3_hyde:.0%}")

check([
    (len(hyde_top3) == 3,              "HyDE returns 3 results"),
    (len(set(hyde_top3) | set(vanilla_top3)) > 0, "Results list not empty"),
], exercise_id="03c-1")

---
## 🟡 EXERCISE — Ex 03c-2: Multi-Query + RAG Fusion

Write `multi_query_rag(question, n_queries=3) -> list[str]`.
LLM generates n_queries reframings. Retrieve top-5 for each.
Merge with Reciprocal Rank Fusion (RRF). Compare Recall@3 vs vanilla.

**RRF score formula:** `score(d) = Σ 1 / (rank + 60)` across all result lists

**Auto-check verifies:**
- n_queries different reframings generated
- RRF scores computed (no division by zero)
- Recall@3 compared single vs multi-query

In [ ]:
def multi_query_rag(question: str, n_queries: int = 3, k_per_query: int = 5) -> list:
    """
    Generate n_queries reframings, retrieve k_per_query per reframing,
    merge with Reciprocal Rank Fusion. Returns top-3 doc IDs.
    """
    # TODO: implement
    # 1. Ask LLM to generate n_queries reframings of the question (one per line)
    # 2. For each reframing, query collection for k_per_query results
    # 3. Apply RRF: for each doc, score += 1 / (rank + 60) for each list it appears in
    # 4. Sort by RRF score descending, return top-3 IDs
    raise NotImplementedError("Implement multi_query_rag()")

In [ ]:
try:
    def mq_query_fn(question, k=3):
        return multi_query_rag(question, n_queries=3)[:k]

    r3_mq = recall_at_k(ground_truth_20, 3, mq_query_fn)

    print(f"Vanilla      Recall@3 : {r3_vanilla:.0%}")
    print(f"HyDE         Recall@3 : {r3_hyde:.0%}")
    print(f"Multi-Query  Recall@3 : {r3_mq:.0%}")

    check([
        (isinstance(r3_mq, float),    "Recall@3 computed for multi-query"),
        (0 <= r3_mq <= 1,             "Recall@3 in [0, 1]"),
    ], exercise_id="03c-2")

except NotImplementedError:
    print("⚠️  Implement multi_query_rag() first.")

---
## 🔴 CHALLENGE — Ex 03c-3: Contextual retrieval

For each Wikipedia chunk, generate a 1-sentence context describing where it comes from.
Prepend it before embedding. Re-measure Recall@3. Is it worth the extra cost?

In [ ]:
# Load Wikipedia
try:
    wiki_ds = load_dataset("wikipedia", "20220301.en", split="train").select(range(100))
    wiki_articles = [{"title": row["title"], "text": row["text"][:1500]} for row in wiki_ds]
    print(f"✅ {len(wiki_articles)} Wikipedia articles loaded")
except Exception as e:
    print(f"⚠️  Wikipedia not available ({e}). Using arXiv abstracts as substitute.")
    wiki_articles = [{"title": a["title"], "text": a["text"]} for a in abstracts[:100]]

# Build chunks from Wikipedia
import tiktoken
enc = tiktoken.get_encoding("cl100k_base")

wiki_chunks = []
for art in wiki_articles[:50]:
    # Simple paragraph chunking
    paras = [p.strip() for p in art["text"].split("\n\n") if len(p.strip()) > 50]
    for para in paras[:3]:   # max 3 chunks per article
        wiki_chunks.append({"title": art["title"], "text": para})

print(f"Total wiki chunks: {len(wiki_chunks)}")

# Generate contextual chunks
print("\nGenerating contextual prefixes…")
contextual_chunks = []
for chunk in wiki_chunks[:30]:   # limit for cost/speed
    context = llm(
        f"In one sentence, describe what section this is from:\nTitle: {chunk['title']}\nText: {chunk['text'][:200]}",
        max_tokens=40
    )
    ctx_text  = context + "\n\n" + chunk["text"]
    ctx_tokens = len(enc.encode(ctx_text))
    contextual_chunks.append({"title": chunk["title"], "text": ctx_text, "tokens": ctx_tokens})

# Index contextual chunks in a new collection
try:
    chroma_client.delete_collection("wiki_ctx")
except:
    pass
wiki_collection = chroma_client.create_collection("wiki_ctx", embedding_function=ef)
wiki_collection.add(
    documents=[c["text"] for c in contextual_chunks],
    metadatas=[{"title": c["title"]} for c in contextual_chunks],
    ids=[str(i) for i in range(len(contextual_chunks))],
)

avg_tokens = sum(c["tokens"] for c in contextual_chunks) / len(contextual_chunks)
cost_est   = (len(contextual_chunks) * avg_tokens / 1000) * 0.00025
print(f"✅ {len(contextual_chunks)} contextual chunks indexed, avg {avg_tokens:.0f} tokens")
print(f"   Estimated context-generation cost: ${cost_est:.4f} (Claude Haiku)")
print(f"   Worthwhile if Recall@3 improves by ≥ 5%")

check([
    (all(c["tokens"] <= 512 + 50 for c in contextual_chunks), "Contextual chunks ≤ ~562 tokens"),
    (len(contextual_chunks) >= 10,                             "≥ 10 contextual chunks created"),
    (isinstance(cost_est, float),                              "Cost estimate computed"),
], exercise_id="03c-3")

In [ ]:
progress()

---
## Readiness checklist before moving to 03d
- [ ] HyDE works — hypothetical document generated and embedded
- [ ] Multi-Query with RRF implemented and tested
- [ ] Contextual retrieval tested with Recall@3 delta measured